# WavLM Utterance Extraction - Auto Resume
## Mount Drive first, then Run All
**Output:** `/content/gdrive/MyDrive/wavlm_utterance_safe/`
**Checkpoint:** Every 5 videos
**If disconnected:** Just re-mount Drive and Run All - auto-resumes

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)
print('Drive mounted')

In [ ]:
# Install + imports
!pip install -q transformers datasets torch torchaudio librosa
import os, json, torch, numpy as np
from transformers import WavLMModel
import librosa
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SR = 16000
OUTPUT_DIR = '/content/gdrive/MyDrive/wavlm_utterance_safe'
CHECKPOINT = os.path.join(OUTPUT_DIR, 'checkpoint.json')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Device: {DEVICE}')

In [ ]:
# Load model
wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base-plus').to(DEVICE)
wavlm.eval()
print('Model loaded')

In [ ]:
# Load utterances
!wget -q -O /tmp/utt.jsonl.gz https://raw.githubusercontent.com/Das-rebel/chuck-audio-notebooks/main/utterances_clean.jsonl.gz
!gunzip -f /tmp/utt.jsonl.gz
utts = [json.loads(l) for l in open('/tmp/utt.jsonl')]
video_utts = {}
for u in utts:
    video_utts.setdefault(u['video_id'], []).append(u)
print(f'Videos: {len(video_utts)}')

In [ ]:
# Find audio files
audio_files = {}
for root, dirs, files in os.walk('/content/gdrive/MyDrive/chuckle_audio_all'):
    for f in files:
        if f.endswith(('.mp3', '.wav')):
            audio_files.setdefault(os.path.splitext(f)[0], os.path.join(root, f))
for base in ['/content/gdrive/MyDrive/audio', '/content/gdrive/MyDrive/chuckle_audio']:
    if os.path.exists(base):
        for root, dirs, files in os.walk(base):
            for f in files:
                if f.endswith(('.mp3', '.wav')):
                    audio_files.setdefault(os.path.splitext(f)[0], os.path.join(root, f))
print(f'Audio files: {len(audio_files)}')

In [ ]:
# Load checkpoint (auto-resume)
if os.path.exists(CHECKPOINT):
    ckpt = json.load(open(CHECKPOINT))
    extracted = set(ckpt['extracted_ids'])
    errors = ckpt['errors']
    print(f'Resuming: {len(extracted)} done, {len(errors)} errors')
else:
    extracted, errors = set(), []
    print('Starting fresh')

vids = [v for v in video_utts if v in audio_files and v not in extracted]
print(f'To process: {len(vids)} videos')

In [ ]:
# Extract function
def extract(seg, net):
    if len(seg) / SR < 0.1: return None
    seg = seg.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = net(seg)
        return out.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

extracted = list(extracted)
print(f'Processing {len(vids)} videos...')

for i, vid in enumerate(tqdm(vids)):
    try:
        audio, _ = librosa.load(audio_files[vid], sr=SR)
        audio_t = torch.tensor(audio, dtype=torch.float32)
        del audio
        results = []
        for u in video_utts[vid]:
            s = int(float(u['start']) * SR)
            e = int(float(u['end']) * SR)
            emb = extract(audio_t[s:e], wavlm)
            if emb is not None:
                results.append({'start': float(u['start']), 'end': float(u['end']), 'embedding': emb.tolist()})
        with open(os.path.join(OUTPUT_DIR, f'{vid}.json'), 'w') as f:
            json.dump({'video_id': vid, 'embeddings': results}, f)
        extracted.append(vid)
        del audio_t
    except Exception as e:
        errors.append({'video_id': vid, 'error': str(e)})
        print(f'Error {vid}: {e}')
    
    torch.cuda.empty_cache()
    
    if (i+1) % 5 == 0:
        json.dump({'extracted_ids': extracted, 'errors': errors}, open(CHECKPOINT, 'w'))
        tqdm.set_postfix(tqdm, extracted=len(extracted), errors=len(errors))

json.dump({'extracted_ids': extracted, 'errors': errors}, open(CHECKPOINT, 'w'))
print(f'Done! Extracted: {len(extracted)}, Errors: {len(errors)}')